[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/10_gqa.ipynb)

# 🔴 Hard: Grouped Query Attention (GQA)

Implement **Grouped Query Attention** — used in LLaMA 2, Mistral, etc. to reduce KV cache size.

Like MHA, but with **fewer KV heads** than Q heads. Each group of Q heads shares the same K/V head.

### Signature
```python
class GroupQueryAttention:
    def __init__(self, d_model: int, num_heads: int, num_kv_heads: int): ...
    def forward(self, x) -> torch.Tensor:  # self-attention
```

### Requirements
- `self.W_q`: `nn.Linear(d_model, d_model)` — full Q projection
- `self.W_k`: `nn.Linear(d_model, num_kv_heads * d_k)` — reduced K projection
- `self.W_v`: `nn.Linear(d_model, num_kv_heads * d_k)` — reduced V projection
- `self.W_o`: `nn.Linear(d_model, d_model)` — output projection
- `d_k = d_model // num_heads`
- Expand KV heads with `repeat_interleave` to match Q heads
- When `num_kv_heads == num_heads`, should behave like standard MHA

In [2]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [5]:
import torch
import torch.nn as nn
import math
# help(torch.repeat_interleave)

In [28]:
# ✏️ YOUR IMPLEMENTATION HERE

class GroupQueryAttention:
    def __init__(self, d_model, num_heads, num_kv_heads):
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, num_kv_heads * self.d_k)
        self.W_v = nn.Linear(d_model, num_kv_heads * self.d_k)
        self.W_o = nn.Linear(d_model, d_model)

        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads

    def forward(self, x):
        B, S, D = x.shape
        q = self.W_q(x)
        k = self.W_k(x)
        v = self.W_v(x)
        q = torch.permute(q.view(B, S, self.num_heads, self.d_k), (0, 2, 1, 3)) # B, H, S, D
        k = torch.permute(k.view(B, S, -1, self.d_k), (0, 2, 1, 3))
        v = torch.permute(v.view(B, S, -1, self.d_k), (0, 2, 1, 3))

        r = self.num_heads // self.num_kv_heads

        # print(f"k: {k.shape}")
        k = torch.repeat_interleave(k, r, dim=1)
        v = torch.repeat_interleave(v, r, dim=1)

        # print(f"k: {k.shape}, q: {q.shape}")

        attn = q @ k.transpose(3, 2) / math.sqrt(self.d_k)
        y = torch.softmax(attn, dim=-1) @ v
        # print(f"y: {y.shape}, q: {q.shape}, {B}, {S}, {D}")
        y = torch.permute(y, (0, 2, 1, 3)).reshape(B, S, D)
        return self.W_o(y)

In [26]:
# 🧪 Debug
torch.manual_seed(0)
gqa = GroupQueryAttention(d_model=32, num_heads=8, num_kv_heads=2)
print("W_q shape:", gqa.W_q.weight.shape)  # (32, 32)
print("W_k shape:", gqa.W_k.weight.shape)  # (8, 32)  — only 2 KV heads * d_k=4

x = torch.randn(2, 6, 32)
out = gqa.forward(x)
print("Output shape:", out.shape)           # (2, 6, 32)

W_q shape: torch.Size([32, 32])
W_k shape: torch.Size([8, 32])
k: torch.Size([2, 2, 6, 4])
k: torch.Size([2, 8, 6, 4]), q: torch.Size([2, 8, 6, 4])
y: torch.Size([2, 8, 6, 4]), q: torch.Size([2, 8, 6, 4]), 2, 6, 32
Output shape: torch.Size([2, 6, 32])


In [27]:
from torch_judge import check
check('gqa')


🧪 Testing: Grouped Query Attention (Hard)
──────────────────────────────────────────────────
k: torch.Size([2, 2, 6, 4])
k: torch.Size([2, 8, 6, 4]), q: torch.Size([2, 8, 6, 4])
y: torch.Size([2, 8, 6, 4]), q: torch.Size([2, 8, 6, 4]), 2, 6, 32
  ✅ [1/5] Output shape (2.7ms)
  ✅ [2/5] nn.Linear with correct shapes (0.7ms)
k: torch.Size([1, 4, 4, 4])
k: torch.Size([1, 4, 4, 4]), q: torch.Size([1, 4, 4, 4])
y: torch.Size([1, 4, 4, 4]), q: torch.Size([1, 4, 4, 4]), 1, 4, 16
  ✅ [3/5] Degenerates to MHA when kv_heads == heads (1.9ms)
  ✅ [4/5] KV heads are shared correctly (5.4ms)
k: torch.Size([1, 2, 4, 4])
k: torch.Size([1, 4, 4, 4]), q: torch.Size([1, 4, 4, 4])
y: torch.Size([1, 4, 4, 4]), q: torch.Size([1, 4, 4, 4]), 1, 4, 16
  ✅ [5/5] Gradient flow (26.2ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (36.8ms total)
  Progress saved. Run status() to see your dashboard.

